## 1. Import Libraries & Load Dataset

**Tujuan:** Memuat kembali dataset asli (bersih) dari `data/raw/` sebagai basis yang akan sengaja dikacaukan pada notebook ini.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/healthcare-dataset-stroke-data.csv')
print(f"Shape: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head(10)

Shape: 5110 baris, 12 kolom


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
5,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1
6,53882,Male,74.0,1,1,Yes,Private,Rural,70.09,27.4,never smoked,1
7,10434,Female,69.0,0,0,No,Private,Urban,94.39,22.8,never smoked,1
8,27419,Female,59.0,0,0,Yes,Private,Rural,76.15,NaN,Unknown,1
9,60491,Female,78.0,0,0,Yes,Private,Urban,58.57,24.2,Unknown,1


**Insight:** Dataset berhasil dimuat kembali dengan shape yang sama seperti NB01 (5.110 baris, 12 kolom), memastikan kita mulai dari kondisi bersih yang identik sebelum proses corruption.

## 2. Set Random Seed

**Tujuan:** Menetapkan random seed satu kali di awal, agar seluruh proses corruption (pemilihan baris untuk duplikasi, gender, glucose, smoking_status) bersifat reproducible — hasil identik setiap kali notebook dijalankan ulang dari atas.

In [2]:
np.random.seed(42)

## 3. Inject Duplicate Rows

**Tujuan:** Mensimulasikan kesalahan input data yang umum terjadi di dunia nyata — pasien yang sama tercatat dua kali (misal akibat human error saat entry data atau proses migrasi sistem). Baris duplikat dipilih secara acak dari data asli dan ditambahkan kembali ke dataset.

In [3]:
n_duplicates = 20

duplicate_indices = np.random.choice(df.index, size=n_duplicates, replace=False)
duplicate_rows = df.loc[duplicate_indices]

df = pd.concat([df, duplicate_rows], ignore_index=True)

print(f"Shape setelah duplikasi: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Jumlah id duplikat: {df['id'].duplicated().sum()}")

Shape setelah duplikasi: 5130 baris, 12 kolom
Jumlah id duplikat: 20


**Insight:** Berhasil menginjeksi 20 baris duplikat, shape berubah dari 5.110 menjadi 5.130 baris. Terkonfirmasi 20 `id` yang muncul duplikat, sesuai jumlah yang diinject.

## 4. Inject Inkonsistensi Kategorikal (Gender)

**Tujuan:** Mensimulasikan inkonsistensi penulisan yang umum terjadi akibat input manual dari sumber berbeda (misal operator berbeda, sistem yang tidak menyeragamkan format) — kolom `gender` yang seharusnya konsisten "Male" dibuat bervariasi menjadi kapital penuh, huruf kecil semua, dan ada spasi tambahan.

In [4]:
male_indices = df[df['gender'] == 'Male'].index
selected = np.random.choice(male_indices, size=50, replace=False)

upper_idx = selected[:20]
lower_idx = selected[20:40]
trailing_idx = selected[40:50]

df.loc[upper_idx, 'gender'] = 'MALE'
df.loc[lower_idx, 'gender'] = 'male'
df.loc[trailing_idx, 'gender'] = 'Male '

print(df['gender'].value_counts())

gender
Female    3006
Male      2073
male        20
MALE        20
Male        10
Other        1
Name: count, dtype: int64


**Insight:** Kolom `gender` kini memiliki 6 variasi kategori dari yang seharusnya cuma 3 (Male/Female/Other): "Male" (2.045), "Female" (2.994), "MALE" (20), "male" (20), "Male " (10), "Other" (1) — total 50 baris terdampak inkonsistensi penulisan, sesuai target injeksi.

## 5. Inject Error Numerik Sistematis (avg_glucose_level)

**Tujuan:** Mensimulasikan kesalahan sistematis pada input numerik — misalnya kesalahan satuan pengukuran atau bug pada alat/sistem pencatatan medis yang menyebabkan nilai tercatat berkali-kali lipat dari nilai sebenarnya. Sejumlah baris dipilih acak dan nilai `avg_glucose_level`-nya dikalikan 5×.

In [5]:
n_glucose_errors = 15
multiplier = 5
lower_bound = 70  

eligible_indices = df[df['avg_glucose_level'] > lower_bound].index
glucose_error_indices = np.random.choice(eligible_indices, size=n_glucose_errors, replace=False)
df.loc[glucose_error_indices, 'avg_glucose_level'] = df.loc[glucose_error_indices, 'avg_glucose_level'] * multiplier

print(df.loc[glucose_error_indices, 'avg_glucose_level'].sort_values())

341      375.90
1117     404.25
3481     435.90
4599     440.20
2783     440.95
1406     452.10
173      455.10
108      465.25
3358     477.45
5077     532.80
3659     570.45
4953     624.60
3724     764.20
1       1011.05
2493    1017.20
Name: avg_glucose_level, dtype: float64


**Insight:** Berhasil menginjeksi error multiplier 5× pada 15 baris `avg_glucose_level` yang dipilih acak. Nilai natural maksimum di dataset asli adalah 271.74 mg/dL — setelah dikalikan 5, nilai-nilai ini seharusnya melampaui batas plausibel klinis (>300 mg/dL), menciptakan outlier yang secara sengaja "tidak natural" untuk dideteksi di tahap cleaning nanti. 

## 6. Inject Missing Value Tambahan (smoking_status)

**Tujuan:** Mensimulasikan missing value dengan mekanisme yang BERBEDA dari kategori "Unknown" yang sudah ada di data asli. Kategori "Unknown" merepresentasikan MNAR (pasien sengaja tidak mau mengungkap status merokoknya). Missing value yang diinjeksi di sini mensimulasikan kegagalan sistem/human error saat pencatatan (misal field terlewat diisi petugas) — pemilihan barisnya murni acak, tidak bergantung pada nilai aslinya maupun variabel lain, sehingga mekanismenya lebih mendekati **MCAR (Missing Completely At Random)**.

In [6]:
n_missing_smoking = 173

eligible_indices = df[df['smoking_status'] != 'Unknown'].index
missing_smoking_indices = np.random.choice(eligible_indices, size=n_missing_smoking, replace=False)

df.loc[missing_smoking_indices, 'smoking_status'] = np.nan

print(df['smoking_status'].value_counts(dropna=False))

smoking_status
never smoked       1806
Unknown            1554
formerly smoked     846
smokes              751
NaN                 173
Name: count, dtype: int64


**Insight:** Kolom `smoking_status` kini memiliki dua bentuk representasi "tidak diketahui" yang berbeda mekanismenya: kategori "Unknown" (MNAR, bawaan data asli, 1.544 baris) dan NaN (MCAR, hasil injeksi, 173 baris). Perbedaan ini penting dipertahankan saat data cleaning nanti, karena strategi penanganan MNAR dan MCAR seharusnya tidak disamakan.

## 7. Verifikasi Hasil Corruption

**Tujuan:** Memvalidasi bahwa seluruh proses corruption (Section 3-6) berhasil diterapkan sesuai target yang direncanakan, sebelum dataset disimpan sebagai file baru di Section 8.

In [8]:
print("=== VERIFIKASI HASIL CORRUPTION ===\n")

print(f"Shape akhir: {df.shape[0]} baris, {df.shape[1]} kolom\n")

print(f"Jumlah id duplikat: {df['id'].duplicated().sum()}\n")

print("Distribusi gender:")
print(df['gender'].value_counts())
print()

print(f"Baris avg_glucose_level > 300: {(df['avg_glucose_level'] > 300).sum()}\n")

print(f"Missing value bmi                  : {df['bmi'].isnull().sum()}")
print(f"Missing value smoking_status (NaN) : {df['smoking_status'].isnull().sum()}")
print(f"Kategori 'Unknown' smoking_status  : {(df['smoking_status']=='Unknown').sum()}")

=== VERIFIKASI HASIL CORRUPTION ===

Shape akhir: 5130 baris, 12 kolom

Jumlah id duplikat: 20

Distribusi gender:
gender
Female    3006
Male      2073
male        20
MALE        20
Male        10
Other        1
Name: count, dtype: int64

Baris avg_glucose_level > 300: 15

Missing value bmi                  : 202
Missing value smoking_status (NaN) : 173
Kategori 'Unknown' smoking_status  : 1554


**Insight:** Seluruh proses corruption terverifikasi sesuai target: 20 baris duplikat, 50 baris inkonsistensi gender, 15 baris error glucose (>300 mg/dL), dan 173 missing value baru pada `smoking_status` (terpisah dari 1.544 kategori "Unknown" bawaan asli). Dataset siap disimpan sebagai file terpisah untuk tahap cleaning di NB03.

## 8. Simpan Dataset Corrupted

**Tujuan:** Menyimpan dataset yang sudah sengaja dikacaukan (Section 3-6) ke file terpisah di `data/processed/`, sebagai input untuk tahap data cleaning di `03_data_cleaning.ipynb`.

In [9]:
df.to_csv('../data/processed/stroke_corruption.csv', index=False)

print("File berhasil disimpan: data/processed/stroke_corruption.csv")

# Verifikasi: baca ulang file yang baru disimpan, pastikan shape-nya cocok
check = pd.read_csv('../data/processed/stroke_corruption.csv')
print(f"Verifikasi shape hasil baca ulang: {check.shape[0]} baris, {check.shape[1]} kolom")

File berhasil disimpan: data/processed/stroke_corruption.csv
Verifikasi shape hasil baca ulang: 5130 baris, 12 kolom


**Insight:** Dataset hasil corruption berhasil disimpan sebagai `stroke_corruption.csv` (5.130 baris, 12 kolom) di `data/processed/`. File ini akan menjadi titik awal proses data cleaning pada notebook berikutnya, dengan seluruh jenis "kerusakan" yang telah diverifikasi pada Section 7: 20 baris duplikat, 50 baris inkonsistensi gender, 15 baris error glucose (multiplier ×5, threshold 70-300 berbasis referensi klinis ADA), dan 173 missing value baru pada `smoking_status`.